# 1. Імпорт бібліотек

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, fbeta_score

# 2. Конфігурація та завантаження даних

In [ ]:
FILE_PATH = "../../data/final_dataset.csv"
TEXT_COLUMN = "text"
LABEL_COLUMN = "fake"

print(f"Завантаження даних з {FILE_PATH}...")
df = pd.read_csv(FILE_PATH, encoding='utf-8')

df = df.rename(columns={LABEL_COLUMN: 'label'})
df = df[[TEXT_COLUMN, 'label']]
df = df.dropna(subset=[TEXT_COLUMN, 'label'])

print("Дані завантажено:")
print(df.info())

# 3. Розділення даних (70/15/15)

In [ ]:
# 70%  / 30%
df_train, df_temp = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])

# 15% / 15%
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp['label'])

X_train = df_train[TEXT_COLUMN]
y_train = df_train['label']

X_val = df_val[TEXT_COLUMN]
y_val = df_val['label']

X_test = df_test[TEXT_COLUMN]
y_test = df_test['label']

print(f"Навчальна вибірка: {len(X_train)} прикладів")
print(f"Валідаційна вибірка: {len(X_val)} прикладів")
print(f"Тестова вибірка:    {len(X_test)} прикладів")

# 4. Векторизація TF-IDF

In [ ]:
print("Навчання TF-IDF векторайзера...")

# Використовуємо параметри зі звіту: n-грами від 1 до 3, макс. 10 тис. ознак
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=10000
)

# Вчимо словник ТІЛЬКИ на навчальних даних
X_train_tfidf = vectorizer.fit_transform(X_train)

# Трансформуємо валідаційні та тестові дані
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Векторизація завершена. Розмірність матриці: {X_train_tfidf.shape}")

# 5. Навчання моделі

In [ ]:
print("Навчання моделі...")

model = LogisticRegression(
    random_state=42, 
    max_iter=100
)
model.fit(X_train_tfidf, y_train)

print("Модель навчена.")

# 6. Оцінка на валідаційній вибірці

In [ ]:
print(" ОЦІНКА НА ВАЛІДАЦІЙНІЙ ВИБІРЦІ ".center(50, "="))

y_val_pred = model.predict(X_val_tfidf)

print(f"Accuracy:  {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred):.4f}")
print(f"F2-score:  {fbeta_score(y_val, y_val_pred, beta=2):.4f}")

# 7. Фінальна оцінка на тестовій вибірці

In [ ]:
print(" ОЦІНКА НА ТЕСТОВІЙ ВИБІРЦІ ".center(50, "="))

y_test_pred = model.predict(X_test_tfidf)

print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"F2-score:  {fbeta_score(y_test, y_test_pred, beta=2):.4f}")

# 8. Перевірка на довільних відгуках

In [ ]:
labels_map = {0: "Автентичний", 1: "Неавтентичний (Фейк)"}

def predict_review(text):
    print("-" * 30)
    print(f"Відгук: '{text}'")

    # 1. Векторизуємо текст за допомогою навченого векторайзера
    text_tfidf = vectorizer.transform([text])

    # 2. Отримуємо прогноз
    prediction = model.predict(text_tfidf)[0]
    
    # 3. Отримуємо ймовірності
    probabilities = model.predict_proba(text_tfidf)[0]
    
    print(f"Прогноз: {labels_map[prediction]} (Клас: {prediction})")
    print(f"Ймовірності:")
    print(f"  {labels_map[0]}: {probabilities[0]:.2%}")
    print(f"  {labels_map[1]}: {probabilities[1]:.2%}")